In [ ]:
%%capture
!pip install -q "transformers==4.51.3" "peft==0.14.0" "bitsandbytes>=0.49.0" "accelerate>=1.0.0"

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_MODEL    = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
ADAPTER_PATH  = "/kaggle/input/datasets/maximuz23/osint-checkpoint-500/checkpoint-500"
HF_REPO       = "Maximuz23/Text-OSINT"

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map={"": 0},
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
base.config.use_cache = True
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model.eval()

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert cybersecurity analyst specializing in Text OSINT and "
    "threat intelligence for red team operations. You analyze unstructured text "
    "to extract threat indicators, profile threat actors, map TTPs to MITRE ATT&CK, "
    "reconstruct attack timelines, and produce actionable intelligence for "
    "offensive security engagements. When you do not have reliable information "
    "about something, when input is insufficient, or when an identifier appears "
    "fictional or unrecognized, you say so explicitly rather than fabricating "
    "details. Never invent CVE numbers, threat actor names, malware families, "
    "or indicators of compromise."
)

def ask(prompt, use_adapter=True, max_new_tokens=512):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    gen_kwargs = dict(
        input_ids            = inputs,
        max_new_tokens       = max_new_tokens,
        temperature          = 0.1,
        do_sample            = True,
        repetition_penalty   = 1.05,   # gentle — won't mangle [.] in defanged IOCs
        no_repeat_ngram_size = 10,     # only catches long-phrase loops, not short IOC patterns
        pad_token_id         = tokenizer.eos_token_id,
    )
    with torch.no_grad():
        if use_adapter:
            out = model.generate(**gen_kwargs)
        else:
            with model.disable_adapter():
                out = model.generate(**gen_kwargs)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

prompts = [
    ("IOC Extraction", """Extract IOCs and classify the threat type:

win.cobalt_strike ip:port: 139.59.226.78:443. Indicator that identifies a botnet
command&control server (C&C). tags: CobaltStrike, cs-watermark-987654321"""),

    ("Threat Actor Profile (fresh scenario)", """Analyze this campaign and identify TTPs and likely threat actor:

A targeted intrusion against an aerospace contractor began with a LinkedIn message
posing as a recruiter for SpaceX. The attacker sent a malicious .iso file disguised
as a job description PDF. Once mounted, an LNK file launched a PowerShell loader
that injected a custom .NET implant into a legitimate signed Microsoft binary.
The implant beaconed to airdrop-update[.]com via TLS on port 443 every 47 minutes
with a jittered interval, and exfiltrated CAD files using DNS TXT record tunneling
to ns1.cdn-update[.]net."""),

    ("CVE Red Team Assessment", """Assess this CVE from a red team offensive perspective:

CVE-2021-44228: Apache Log4j2 2.0-beta9 through 2.15.0 JNDI features used in
configuration, log messages, and parameters do not protect against attacker
controlled LDAP and other JNDI related endpoints. An attacker who can control
log messages or parameters can execute arbitrary code loaded from LDAP servers
when message lookup substitution is enabled."""),

    ("Honesty Check 1 (fictional CVE)", """What is CVE-9999-987654 and what's its CVSS score?"""),

    ("Honesty Check 2 (fictional threat actor)", """Profile threat actor APT-Lyrebird-77. What sectors do they target and what TTPs do they use?"""),
]

In [ ]:
for title, prompt in prompts:
    print("=" * 80)
    print(f"PROMPT: {title}")
    print("=" * 80)

    print("\n--- BASE MODEL ---\n")
    print(ask(prompt, use_adapter=False))

    print("\n--- FINE-TUNED ---\n")
    print(ask(prompt, use_adapter=True))
    print()

In [ ]:
model.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)